# 05: Regime Analysis

This final notebook explores the strategy's sensitivity to broad market regimes, the trade-offs between **Short** and **Long** lookback windows, and the structural failure modes identified during the 2018–2020 period.

## 1. The 2018–2020 Stress Period Recap

As identified in [04: Out-of-Sample Results](04_oos_results.ipynb), the 2018–2020 period was a high-dispersion regime in which the strategy's performance characteristics shifted. This shift was characterized by a change in exit dynamics:
- **Mean Reversion Success:** Dropped from 55.6% (In-Sample) and 55.3% (Recovery) to 41.5% in the stress period.
- **Stop Loss/Guard Frequency:** Spiked as idiosyncratic shocks decoupled historical pairs.

### Universal Failure: Parameters vs. Regimes
To confirm that this performance was structural rather than parametric, each fold's in-sample optimal parameter set was evaluated over the out-of-sample period.

| Metric | Stress Period (2018-20) | Recovery Period (2024-25) |
|---|---|---|
| **Mean OOS Sharpe** | -0.82 | +0.74 |
| **Best Aggregate Sharpe** | 0.05 | 0.65 |
| **Primary Failure Mode** | Structural Decoupling | Consistent Reversion |

No parameter combination—Lookback <= 45 or Lookback > 45—was sufficient to achieve profitability during the fundamental decoupling of 2018–2020. This reinforces that the strategy is a model designed for stable sector regimes.

## 2. Lookback Duration: The Window Length Trade-off

Despite universal performance degradation in decoupled regimes, research shows a clear sensitivity to the residual lookback window length.

Shorter residual lookbacks (<= 45 days) react faster to new data but are prone to noise, while longer lookbacks (> 45 days) perform better in persistent, low-volatility environments.
![Lookback Sensitivity](../results/regime_analysis//lookback_sensitivity_bars.png)

For Mega-cap equities, **Longer Lookbacks generally perform better**. The structural relationships between firms are driven by long-term fundamental sector drivers that are obscured by short-term noise when using small lookback windows.

## 3. Hostile Regime Characteristics: Why and How the Strategy Fails

Through diagnostic research and the comparison of Out-of-Sample (OOS) exit reasons, we have identified how the strategy's statistical engine reacts to different regimes:

### Failure Composition: In-Sample vs. OOS
![Exit Distribution Distribution](../results/regime_analysis/oos_exit_distribution_bars.png)

| Exit Reason | In-Sample (2020-24) | Recovery (2024-25) | Stress (2018-20) | Interpretation |
|---|---|---|---|---|
| **Signal** | 55.6% | 55.3% | 41.5% | Successful mean reversion. |
| **Stop Loss** | 18.5% | 21.1% | 35.4% | Idiosyncratic Shocks Spikes in stress regimes. |
| **Guard Trip** | 0.9% | 13.2% | 6.2% | Structural Decoupling Proactive safety valve. |
| **Max Hold Exit** | 13.0% | 7.9% | 10.8% | Position timed out with a loss. |
| **Timeout** | 12.0% | 2.6% | 6.2% | Duration-based exit (neutral/win). |

## 4. Deep Dive: Forensics (EZ2.2_EXZ1.0_HRT0.8_RV90)
While the **EZ2.2_EXZ1.0_HRT0.8_RV90** set is the top-ranked set in In-Sample optimization, multi-period tests reveal structural vulnerabilities dependent on the market regime.

***Note:** All PnL and % return figures below are relative to the **specific notional value** of that trade (typically $180,000 per leg) and do not represent a percentage of the total allocated portfolio capital.*

#### 1. 2018–2020 Stress Period
- **Worst Trades:** `HD-LOW` (-7.6%, 2018 Q2), `HD-SBUX` (-7.4%, 2018 Q2), `LLY-UNH` (-7.2%).
- **Stop Loss Hotspots:** **Fold 5** (2019 Q2) and **Fold 4** (2019 Q1).
- **Failure Mode:** Structural decoupling in Retail(Staples and Discretionary). The 90-day lookback was too static to adapt to these shifts. 2 of our 3 worst performing pairs include HD, which implies concentration risk.
- **Empirical Observation:** **Shorter Lookbacks (RV30/RV60)** performed better in these folds.

#### 2. 2024–2025 OOS Period
- **Worst Trades:** `AAPL-INTC` (-26.2%), `ABT-AMGN` (-7.0%), `NVDA-TXN` (-4.7%).
- **Stop Loss Hotspots:** **Fold 4** (2025 Q1) and **Fold 7** (2025 Q4).
- **Failure Mode:** Severe idiosyncratic rapid decoupling events in Tech and Healthcare, exemplified by the Feb 2025 AAPL-INTC -26% tail event.
- **Empirical Observation:** **Shorter Lookbacks (RV30/RV60)** and **Entry Z 1.8** was able to circumvent the AAPL-INTC trade.


## 5. Market Neutrality & Beta Analysis

A rigorous regression analysis was performed against the S&P 500 (SPY) for the 2024–2025 period to quantify the strategy's market sensitivity. The results confirm that the strategy is structurally market-neutral. 

### Summary Statistics
| Metric | Full Period | Active Days Only |
| :--- | :--- | :--- |
| **Market Beta** | 0.0030 | 0.0111 |
| **Correlation** | 0.0110 | 0.0205 |
| **Annualized Alpha** | +2.89% | +8.33% |
| **R-Squared** | 0.0001 | 0.0004 |

### Fold-by-Fold Breakdown
| Label | Beta | Alpha (Ann.) | Correlation | Count |
| :--- | :--- | :--- | :--- | :--- |
| Fold 0 | 0.0173 | +6.88% | 0.0559 | 61 |
| Fold 1 | 0.0143 | +5.30% | 0.0341 | 63 |
| Fold 2 | -0.0176 | +2.11% | -0.1090 | 64 |
| Fold 3 | -0.0264 | +10.57% | -0.1185 | 64 |
| Fold 4 | 0.0440 | -7.70% | 0.0778 | 60 |
| Fold 5 | 0.0046 | +10.85% | 0.0528 | 62 |
| Fold 6 | -0.0208 | +0.47% | -0.1071 | 64 |
| Fold 7 | -0.0333 | -2.40% | -0.0922 | 64 |

**Interpretation:** The strategy maintains a Beta near zero across all time segments. Market movements explain less than 0.05% of the performance variance, indicating that returns are almost entirely driven by idiosyncratic pair alpha.

## 6. Known Limitations & Future Work: VIX Velocity

This system represents a robust baseline for mean-reversion trading. Future work to extend its applicability to hostile regimes includes:

- **VIX Velocity Filters:** Preliminary analysis suggests a possible correlation between rapid VIX expansion (e.g., >30% in 5 days) and strategy failure. Formalizing this as a robust predictive indicator would quantify the "Vol-of-Vol" threshold at which mean-reversion probabilities drop significantly.
- **Regime Identification:** It was documented in [03: Walk forward results](03_walk_forward_results.ipynb) that 40% hedge ratio drift from selection period could signal an unfavourable regime. This threshold could be used to identify regimes and parameters can be adjusted accordingly.
- **Dynamic Lookbacks:** Automatically shortening the OLS lookback window (e.g., switching from 90-day to 30-day) during periods of high VIX expansion to adapt faster to regime shifts.
- **Adaptive Sizing:** De-leveraging the portfolio when volatility acceleration exceeds historical norms.
- **Position Sizing:** Current system does not include a strategy to size positions. Sizing positions based on spread volatility, hedge ratio drift upon entry could help improve performance.
- **Sector Concentration Guards:** Limiting the total number of pairs per sector to prevent 'Double Jeopardy' when a whole industry experiences a structural shock.
- **News/Earnings Blackout Filter:** Programmatically skipping entries within ± 3 days of an earnings/news announcement to mitigate idiosyncratic 'Gap' risk.

## 7. Conclusion: System Characteristics

The Algorithmic Pairs Trading System is a technical model designed for a specific market regime:

- **Optimal Regime:** Stable sector relationships, moderate volatility, low idiosyncratic news dispersion (e.g., 2024–2026).
- **Hostile Regime:** Fundamental sector decoupling, high idiosyncratic gaps, and structural breaks in cointegration (e.g., 2018–2020).

### Final Summary
The strategy is stability-biased — it excels in persistent, correlated regimes where cointegration holds and mean reversion is consistent. The Hedge Ratio Guard and locked reference logic provide structural protection against gradual drift, but offer limited defense against idiosyncratic gap events. The 2018–2020 stress period demonstrated that this failure mode is regime-structural, not parametric — no parameter combination achieved meaningful profitability. Future extensions should prioritize gap risk mitigation and concentration controls.